# Do Vulnerability Detectors Learn the Bug or the Names?
**TF-IDF with Random Forest versus fine-tuned CodeBERT on C functions**

ARTI 404 Natural Language Processing · IAU (CCSIT) · Section 9MA2 · Group 1
Khalid Waleed Alhilal (leader) · Ali Khalid Albinali · Mishary Waleed AlAwadh

**Question.** Which method detects vulnerable C functions more reliably, and which one loses less accuracy
and macro F1 when identifier names are replaced?

**Input / output.** One C function as text → `Vulnerable` or `Safe` and the probability of `Vulnerable`.

## 1. Setup
Environment: Jupyter (local), Python 3.11, CUDA 12.8, NVIDIA RTX 5080.

* `RUN_FULL_TRAINING = False` (default): loads the saved models (downloaded from the GitHub Release if they
  are missing) and the saved results, then recomputes the seed-42 test metrics from the loaded models and
  checks that they equal `results/metrics_summary.json`.
* `RUN_FULL_TRAINING = True`: reruns every experiment with `scripts/run_experiments.py`
  (about 3 hours on an RTX 5080).

In [ ]:
RUN_FULL_TRAINING = False

import json
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd()
assert (REPO / "src").is_dir(), "Start Jupyter in the repository root folder."
sys.path.insert(0, str(REPO))
RESULTS = REPO / "results"
pd.set_option("display.max_colwidth", 80)


def run(*args: str) -> None:
    """Run one step of scripts/run_experiments.py and show its output."""
    cmd = [sys.executable, "scripts/run_experiments.py", *args]
    print("$ python", " ".join(cmd[1:]))
    subprocess.run(cmd, check=True, cwd=REPO)


def load_json(name: str) -> dict:
    with open(RESULTS / name, encoding="utf-8") as fh:
        return json.load(fh)

In [ ]:
# Package versions used for the reported results
import importlib.metadata as md
import torch

print(f"Python {platform.python_version()} | torch {torch.__version__} | CUDA {torch.version.cuda} | "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)'}")
for pkg in ["transformers", "datasets", "scikit-learn", "pandas", "numpy", "tree-sitter", "tree-sitter-c",
            "scipy", "matplotlib", "joblib"]:
    print(f"  {pkg:<13} {md.version(pkg)}")

In [ ]:
# Saved models: download from the GitHub Release if they are not in models/ yet
if not RUN_FULL_TRAINING:
    from src.assets import ensure_models
    ensure_models()

## 2. Data
CodeXGLUE Defect Detection (Devign): C functions from FFmpeg and QEMU, label 1 = vulnerable.
`data_prep.py` removes comments and extra whitespace and hashes every function, so that copies of the same
function stay in one split only (no test function is seen in training). Functions whose copies carry
different labels are removed.

In [ ]:
from src import data_prep

if RUN_FULL_TRAINING or not (REPO / "data" / "processed" / "test_renamed.parquet").exists():
    run("download")   # Hugging Face -> data/raw/
    run("clean")      # data_prep.py -> data/processed/{train,dev,test}.parquet + results/data_stats.json
    run("rename")     # renaming.py  -> test_renamed.parquet, train_augmented.parquet
    run("rename-extra")  # word-name scheme, variables-only and function-names-only test sets

splits = data_prep.load_splits("data/processed", names=("train", "dev", "test", "test_renamed"))
stats = load_json("data_stats.json")

counts = pd.DataFrame(stats["after_cleaning"]).T
counts.insert(0, "official", pd.Series(stats["official"]))
display(counts)
print("Duplicates:", stats["duplicates"])
t = stats["test_truncated_512"]
print(f"Test functions longer than 512 CodeBERT tokens: {t['count']} ({100 * t['pct']:.2f}%)")

In [ ]:
# One cleaned training example
row = splits["train"].iloc[0]
print("label:", row["label"], "(1 = vulnerable)")
print(row["code"][:600])

### Renamed test set
`renaming.py` parses each function with tree-sitter and replaces user-defined names consistently:
the function's own name and called project functions become `FUN1, FUN2, …`, all other identifiers
`VAR1, VAR2, …`. Keywords, C library calls, struct fields, type names and ALL_CAPS macros are kept, so the
program does the same thing and the label stays valid.

In [ ]:
ren = load_json("renaming_stats.json")
display(pd.Series(ren, name="value").to_frame())

i = splits["test"].index[0]
print("BEFORE:\n", splits["test"].loc[i, "code"][:500])
print("\nAFTER:\n", splits["test_renamed"].loc[i, "code"][:500])

## 3. Method 1 — TF-IDF + Random Forest
`c_tokenizer.py` keeps multi-character operators whole, replaces string and char literals with `<STR>` /
`<CHR>`, and adds the lower-cased parts of every identifier (`dstBuf` → `dstBuf`, `dst`, `buf`).
TF-IDF on unigrams + bigrams feeds a Random Forest. All hyper-parameters are chosen on dev macro F1
(seed 42); the test sets are never used for tuning.

In [ ]:
from src.c_tokenizer import tokenize_c

print(tokenize_c("if (len > n) memcpy(dstBuf, src_ptr, len);"))

In [ ]:
if RUN_FULL_TRAINING:
    run("tfidf-tune")         # 64 settings on dev
    run("tfidf-seeds")        # seeds 42, 123, 2026 with the best settings
    run("length-baseline")    # diagnostic: function length only

grid = pd.read_csv(RESULTS / "tfidf_grid.csv")
print("Best settings (dev macro F1):", load_json("tfidf_best_params.json")["best_params"])
display(grid.head(5))

In [ ]:
# Top 30 features of the seed-42 model, with their kind
display(pd.read_csv(RESULTS / "tfidf_top_features.csv"))

## 4. Method 2 — fine-tuned CodeBERT
`microsoft/codebert-base` (125M parameters) + a classification head. Each function is cut to 512 sub-word
tokens. AdamW with a linear schedule and 10% warm-up, weight decay 0.01, effective batch 32, bf16.
The learning rate is chosen from {1e-5, 2e-5, 3e-5} on dev macro F1 (seed 42), and the best epoch is
kept with early stopping (patience 2).

In [ ]:
if RUN_FULL_TRAINING:
    run("codebert-lr-search")   # seed 42, three learning rates
    run("codebert-seeds")       # seeds 123 and 2026 with the chosen learning rate

display(pd.read_csv(RESULTS / "codebert_lr_search.csv"))
print("Chosen:", load_json("codebert_best_params.json"))

with open(REPO / "models" / "codebert_seed42" / "train_log.json", encoding="utf-8") as fh:
    log = json.load(fh)
display(pd.DataFrame(log["history"]))

## 5. Results
Every number below is read from `results/metrics_summary.json` (mean ± std over seeds 42, 123, 2026).
Drop = original test − renamed test.

In [ ]:
if RUN_FULL_TRAINING:
    run("augment", "--method", "tfidf_rf", "--seeds", "42", "123", "2026")   # original + renamed copies
    run("augment", "--method", "codebert", "--seeds", "42", "123", "2026")
    run("predict-missing")   # score every saved model on every test set
    run("ensemble")          # mean probability of the two methods (evaluation only)
    run("head-tail")         # CodeBERT with first 128 + last 382 tokens for long functions
    run("summarize")

summary = load_json("metrics_summary.json")


def fmt(cell):
    if cell is None:
        return ""
    return f"{100 * cell['mean']:.2f}" + ("" if cell.get("std") is None else f" ± {100 * cell['std']:.2f}")


rows = []
TEST_KEYS = [("original_test", "original test"), ("renamed_test", "renamed test (VARn/FUNn)"),
             ("renamed_words_test", "renamed test (word names)"), ("drop", "drop VARn/FUNn"), ("drop_words", "drop word names")]
for method, name in [("tfidf_rf", "TF-IDF + RF"), ("codebert", "CodeBERT"), ("length_baseline", "Length only (diagnostic)")]:
    block = summary.get(method, {})
    for key, label in TEST_KEYS:
        if key in block:
            rows.append({"method": name, "set": label,
                         **{m: fmt(block[key].get(m)) for m in ["accuracy", "macro_f1", "vuln_recall", "roc_auc"]}})
display(pd.DataFrame(rows))
print("Majority baseline (original test):", summary["majority_baseline"]["original_test"])

In [ ]:
# McNemar's test on the paired seed-42 predictions (TF-IDF + RF vs CodeBERT)
display(pd.DataFrame(summary["mcnemar"]).T)

In [ ]:
for f in ["drop_chart.png", "cm_tfidf_rf_orig.png", "cm_tfidf_rf_ren.png", "cm_codebert_orig.png", "cm_codebert_ren.png"]:
    path = RESULTS / "figures" / f
    if path.exists():
        display(Image(filename=str(path), width=520))

In [ ]:
# Functions longer than 512 tokens (CodeBERT sees only the first 512) vs the rest, seed 42
display(pd.read_csv(RESULTS / "truncation_table.csv"))

In [ ]:
# Planned improvement: training on original + renamed copies (3 seeds)
aug = summary["augmentation"]
rows = []
for method, name in [("tfidf_rf", "TF-IDF + RF"), ("codebert", "CodeBERT"), ("ensemble", "Ensemble")]:
    for key, label in TEST_KEYS:
        if key in aug.get(method, {}):
            rows.append({"method": name + " (augmented)", "set": label,
                         **{m: fmt(aug[method][key].get(m)) for m in ["accuracy", "macro_f1", "vuln_recall", "roc_auc"]}})
display(pd.DataFrame(rows))

### Extra analyses (evaluation only, no new training)
* **Partial renaming:** only variables (`VARn`) or only function names (`FUNn`) are replaced.
* **Ensemble:** mean of P(vulnerable) from TF-IDF + RF and CodeBERT (same seed), threshold 0.5.
* **Head + tail:** CodeBERT reads the first 128 and the last 382 tokens of functions longer than 512 tokens.

In [ ]:
rows = []
for method, name in [("tfidf_rf", "TF-IDF + RF"), ("codebert", "CodeBERT"), ("ensemble", "Ensemble")]:
    block = summary[method]
    for key, label in [("drop", "all names"), ("drop_var_only", "variables only"), ("drop_fun_only", "function names only")]:
        rows.append({"method": name, "renamed": label, **{m: fmt(block[key].get(m)) for m in ["accuracy", "macro_f1", "roc_auc"]}})
print("Drop (original - renamed), percentage points")
display(pd.DataFrame(rows))

In [ ]:
rows = []
for method, name in [("codebert", "CodeBERT"), ("ensemble", "Ensemble")]:
    for key, label in TEST_KEYS[:3]:
        rows.append({"model": name, "set": label, **{m: fmt(summary[method][key].get(m)) for m in ["accuracy", "macro_f1", "vuln_recall"]}})
display(pd.DataFrame(rows))
print("McNemar ensemble vs CodeBERT (seed 42):")
display(pd.DataFrame(summary["mcnemar_ensemble_vs_codebert"]).T)

In [ ]:
t = summary["truncation"]
cut = pd.DataFrame({"first 510 tokens": t["codebert"]["cut"], "first 128 + last 382": t["codebert_head_tail"]["cut"]}).T
print("CodeBERT on the 873 truncated test functions (seed 42)")
display(cut)
print("Full original test, 3 seeds: first 510 ->", fmt(summary["codebert"]["original_test"]["accuracy"]),
      "| head + tail ->", fmt(summary["codebert_head_tail"]["original_test"]["accuracy"]), "(accuracy, %)")

### Reproducibility check
The seed-42 models are loaded from disk and scored again on the original and renamed test sets.
The metrics must match `metrics_summary.json → primary_seed_runs` (tolerance 0.001, i.e. at most a few
predictions may flip between GPU and CPU arithmetic).

In [ ]:
from src import codebert_model, evaluation, tfidf_rf

tf_model = tfidf_rf.load(str(REPO / "models" / "tfidf_rf_seed42.joblib"))
cb_model, cb_tok = codebert_model.load(str(REPO / "models" / "codebert_seed42"))

for split in ["test", "test_renamed"]:
    df = splits[split]
    probs = {"tfidf_rf": tfidf_rf.predict_proba(tf_model, df["code"].tolist()),
             "codebert": codebert_model.predict_proba(cb_model, cb_tok, df["code"].tolist())}
    for method, p in probs.items():
        now = evaluation.compute_metrics(df["label"].values, (p >= 0.5).astype(int))
        saved = summary["primary_seed_runs"][method][split]
        for k in ["accuracy", "macro_f1", "vuln_recall"]:
            assert abs(now[k] - saved[k]) <= 1e-3, (method, split, k, now[k], saved[k])
        print(f"{method:<9} {split:<13} OK  " + "  ".join(f"{k}={now[k]:.4f}" for k in ["accuracy", "macro_f1", "vuln_recall"]))

## 6. Try it
Put one C function in `code` and run the cell. The input gets the same cleaning as the training data and
is classified by the saved models (no retraining). The selected model (higher mean dev macro F1) is
printed first, the other one for comparison.

In [ ]:
from src.predict import try_it

# New vulnerable example: the length comes from the input and is never checked against the buffer size
code = """
int parse_header(const unsigned char *data, int size, char *name)
{
    int len = data[0];
    memcpy(name, data + 1, len);
    name[len] = 0;
    return len + 1;
}
"""
try_it(code)

In [ ]:
# New safe example: the length is checked against both the input size and the buffer size
code = """
int parse_header(const unsigned char *data, int size, char *name, int name_size)
{
    int len;
    if (size < 1)
        return -1;
    len = data[0];
    if (len >= name_size || len > size - 1)
        return -1;
    memcpy(name, data + 1, len);
    name[len] = 0;
    return len + 1;
}
"""
try_it(code)

In [ ]:
# Blank input
code = "   "
try_it(code)

### Original examples (written by the team; not part of the test score)

In [ ]:
with open(REPO / "data" / "original_examples.json", encoding="utf-8") as fh:
    examples = pd.DataFrame(json.load(fh))

clean = [data_prep.clean_code(c) for c in examples["code"]]
examples["tfidf_rf_prob"] = tfidf_rf.predict_proba(tf_model, clean).round(3)
examples["codebert_prob"] = codebert_model.predict_proba(cb_model, cb_tok, clean).round(3)
for m in ["tfidf_rf", "codebert"]:
    examples[f"{m}_correct"] = (examples[f"{m}_prob"] >= 0.5).astype(int) == examples["label"]
display(examples[["author", "id", "label", "tfidf_rf_prob", "codebert_prob", "tfidf_rf_correct", "codebert_correct", "why"]])

## 7. Sources
**Data and models**
- CodeXGLUE Defect Detection (Devign), Hugging Face `google/code_x_glue_cc_defect_detection`.
- `microsoft/codebert-base`, Hugging Face Transformers.
- tree-sitter and tree-sitter-c (C parser used for renaming).

**Libraries**: PyTorch, Transformers, Datasets, scikit-learn, pandas, NumPy, SciPy, statsmodels, Matplotlib, joblib.

**Papers**
1. S. Lu et al., "CodeXGLUE: A machine learning benchmark dataset for code understanding and generation," Proc. NeurIPS Datasets and Benchmarks Track, 2021.
2. Y. Zhou, S. Liu, J. Siow, X. Du, and Y. Liu, "Devign: Effective vulnerability identification by learning comprehensive program semantics via graph neural networks," Proc. NeurIPS, 2019.
3. R. Russell et al., "Automated vulnerability detection in source code using deep representation learning," Proc. IEEE ICMLA, 2018, pp. 757–762.
4. Z. Feng et al., "CodeBERT: A pre-trained model for programming and natural languages," Findings of EMNLP, 2020.
5. Z. Li et al., "VulDeePecker: A deep learning-based system for vulnerability detection," Proc. NDSS, 2018.
6. N. Risse and M. Böhme, "Uncovering the limits of machine learning for automatic vulnerability detection," Proc. USENIX Security, 2024.
7. M. Fu and C. Tantithamthavorn, "LineVul: A transformer-based line-level vulnerability prediction," Proc. MSR, 2022.
8. S. Chakraborty, R. Krishna, Y. Ding, and B. Ray, "Deep learning based vulnerability detection: Are we there yet?" IEEE Trans. Softw. Eng., vol. 48, no. 9, pp. 3280–3296, 2022.